# Notebook 1: ITS_LIVE Velocity Ground Truth (NSIDC-0766)


In [ ]:
import os, glob, gc, re, json, warnings
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
from tqdm import tqdm

warnings.filterwarnings('ignore')

import rasterio
from rasterio.crs import CRS

ITSLIVE_DIR   = r"./data/NSIDC-0766"
OUTPUT_DIR    = "./processed_velocity_npz"
FIGURES_DIR   = "./figures_velocity"
TARGET_RES    = 200  # meters

ROI_BOUNDS = {
    'min_lon': -31.904297, 'max_lon': -17.929688,
    'min_lat': 76.679785,  'max_lat': 80.253391,
}

for d in [OUTPUT_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"ITS_LIVE dir: {ITSLIVE_DIR}")
print(f"Output dir:   {OUTPUT_DIR}")


## 1) Scan & Catalog Velocity Files — Grouped by Cycle


In [ ]:
def parse_ddmonyy(date_str):
    """Parse DDMonYY format like '01Apr16' -> datetime(2016, 4, 1)."""
    try:
        return datetime.strptime(date_str, '%d%b%y')
    except ValueError:
        return None

def scan_and_group_cycles(itslive_dir):
    """
    Scan NSIDC-0766 directory and group files by cycle.
    Returns dict: { cycle_key: { 'date_start': dt, 'date_end': dt,
                                  'vx': path, 'vy': path, 'ex': path, 'ey': path } }
    """
    pattern = re.compile(
        r'GL_vel_mosaic_s1cycle_'
        r'(\d{2}[A-Za-z]{3}\d{2})_'
        r'(\d{2}[A-Za-z]{3}\d{2})_'
        r'(browse|dT|ex|ey|vv|vx|vy)_'
        r'v02\.0\.tif$'
    )
    
    cycles = defaultdict(dict)
    tif_files = sorted(glob.glob(os.path.join(itslive_dir, '*.tif')))
    print(f"Total .tif files found: {len(tif_files)}")
    
    needed_types = {'vx', 'vy', 'ex', 'ey'}
    
    for fp in tif_files:
        m = pattern.search(Path(fp).name)
        if not m:
            continue
        d1_str, d2_str, ftype = m.group(1), m.group(2), m.group(3)
        if ftype not in needed_types:
            continue
        d1 = parse_ddmonyy(d1_str)
        d2 = parse_ddmonyy(d2_str)
        if d1 is None or d2 is None:
            print(f"  WARNING: date parse failed for {Path(fp).name}")
            continue
        cycle_key = f"{d1_str}_{d2_str}"
        cycles[cycle_key]['date_start'] = d1
        cycles[cycle_key]['date_end'] = d2
        cycles[cycle_key][ftype] = fp
    
    complete = {}
    incomplete = []
    for key, info in cycles.items():
        if 'vx' in info and 'vy' in info:
            complete[key] = info
        else:
            incomplete.append((key, list(info.keys())))
    
    complete = dict(sorted(complete.items(), key=lambda x: x[1]['date_start']))
    
    print(f"\nComplete cycles (vx+vy): {len(complete)}")
    if incomplete:
        print(f"Incomplete cycles: {len(incomplete)}")
    
    dates = [v['date_start'] for v in complete.values()]
    if dates:
        print(f"Date range: {min(dates).strftime('%Y-%m-%d')} → {max(dates).strftime('%Y-%m-%d')}")
    
    sample_key = list(complete.keys())[0]
    print(f"\nSample cycle '{sample_key}':")
    for ftype in ['vx', 'vy', 'ex', 'ey']:
        if ftype in complete[sample_key]:
            fp = complete[sample_key][ftype]
            sz = os.path.getsize(fp) / 1e6
            print(f"  {ftype}: {Path(fp).name} ({sz:.1f} MB)")
    
    return complete

cycle_catalog = scan_and_group_cycles(ITSLIVE_DIR)


## 2) Load & Inspect a Single Cycle (Full Raster)


In [ ]:
def load_cycle_full(cycle_info, verbose=True):
    '''
    Load vx, vy, ex, ey for one cycle — cropped to ROI_BOUNDS.
    Returns dict with arrays + metadata, or None if all-zero/empty.
    '''
    from rasterio.warp import transform_bounds
    from rasterio.windows import from_bounds
    import rasterio
    import numpy as np

    result = {}
    
    with rasterio.open(cycle_info['vx']) as src:
        orig_bounds = (ROI_BOUNDS['min_lon'], ROI_BOUNDS['min_lat'], ROI_BOUNDS['max_lon'], ROI_BOUNDS['max_lat'])
        proj_bounds = transform_bounds('EPSG:4326', src.crs, *orig_bounds)
        window = from_bounds(*proj_bounds, transform=src.transform)
        
        window = window.round_lengths().round_offsets()
        
        win_transform = src.window_transform(window)
        
        if verbose:
            print(f"Original Raster: {src.width}x{src.height}, CRS: {src.crs}")
            print(f"Resolution: {src.res}, NoData: {src.nodata}")
            print(f"Projected ROI Bounds: {proj_bounds}")
            print(f"Window: {window}")
        
        result['transform'] = win_transform
        result['crs'] = str(src.crs)
        result['nodata'] = src.nodata
        result['bounds'] = rasterio.windows.bounds(window, src.transform)
        result['shape'] = (window.height, window.width)
        result['window'] = window
    
    for ftype in ['vx', 'vy', 'ex', 'ey']:
        fp = cycle_info.get(ftype)
        if fp is None:
            if verbose:
                print(f"  {ftype}: not available")
            continue
        
        with rasterio.open(fp) as src:
            arr = src.read(1, window=result['window']).astype(np.float32)
            nodata = src.nodata
        
        if nodata is not None:
            arr[arr == nodata] = np.nan
        
        result[ftype] = arr
        
        if verbose:
            valid = ~np.isnan(arr)
            if valid.any():
                print(f"  {ftype}: shape={arr.shape}, "
                      f"valid={valid.sum():,}/{arr.size:,} ({valid.sum()/arr.size*100:.1f}%), "
                      f"range=[{np.nanmin(arr):.1f}, {np.nanmax(arr):.1f}]")
            else:
                print(f"  {ftype}: shape={arr.shape}, ALL NaN/nodata")
    
    vx = result.get('vx')
    vy = result.get('vy')
    if vx is None or vy is None:
        if verbose: print("  ⚠ Missing vx or vy")
        return None
    
    both_valid = ~np.isnan(vx) & ~np.isnan(vy)
    if both_valid.sum() == 0:
        if verbose: print("  ⚠ ALL NODATA — skipping")
        return None
    
    if not np.any(vx[both_valid] != 0) and not np.any(vy[both_valid] != 0):
        if verbose: print("  ⚠ ALL ZEROS — skipping")
        return None
    
    result['date_start'] = cycle_info['date_start']
    result['date_end'] = cycle_info['date_end']
    del result['window']
    return result


sample_key = None
sample_data = None
for key, info in cycle_catalog.items():
    if info['date_start'].year >= 2018:
        print(f"{'='*60}")
        print(f"Inspecting cycle: {key}")
        print(f"{'='*60}")
        sample_data = load_cycle_full(info)
        if sample_data is not None:
            sample_key = key
            break

if sample_data is None:
    for key, info in cycle_catalog.items():
        sample_data = load_cycle_full(info)
        if sample_data is not None:
            sample_key = key
            break

if sample_data:
    print(f"\n✓ Loaded cycle: {sample_key}")
    print(f"  Shape: {sample_data['vx'].shape}")
else:
    print("✗ No valid cycle found.")


## 3) Spatial Completeness Analysis


In [ ]:
def analyze_spatial_coverage(data, title="Velocity Coverage"):
    if data is None or 'vx' not in data:
        print("No valid data."); return None
    
    vx, vy = data['vx'], data['vy']
    valid = ~np.isnan(vx) & ~np.isnan(vy) & (np.abs(vx) < 50000) & (np.abs(vy) < 50000)
    
    total_px = vx.size
    valid_px = valid.sum()
    print(f"Valid pixels: {valid_px:,}/{total_px:,} ({valid_px/total_px*100:.1f}%)")
    
    step = max(1, max(vx.shape) // 2000)
    vx_s = vx[::step, ::step]
    vy_s = vy[::step, ::step]
    valid_s = valid[::step, ::step]
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(title, fontsize=14)
    
    axes[0,0].imshow(valid_s.astype(float), cmap='RdYlGn', vmin=0, vmax=1)
    axes[0,0].set_title(f'Valid mask ({valid_px/total_px*100:.1f}%)')
    
    im = axes[0,1].imshow(np.where(valid_s, vx_s, np.nan), cmap='RdBu_r', vmin=-5000, vmax=5000)
    axes[0,1].set_title('vx (m/yr)'); plt.colorbar(im, ax=axes[0,1])
    
    if valid_px > 0:
        axes[0,2].hist(vx[valid].ravel()[::max(1, valid_px//100000)], bins=200, log=True,
                       color='steelblue', edgecolor='none')
        axes[0,2].set_title('vx distribution'); axes[0,2].set_xlabel('m/yr')
    
    im = axes[1,0].imshow(np.where(valid_s, vy_s, np.nan), cmap='RdBu_r', vmin=-5000, vmax=5000)
    axes[1,0].set_title('vy (m/yr)'); plt.colorbar(im, ax=axes[1,0])
    
    speed_s = np.sqrt(np.where(valid_s, vx_s, 0)**2 + np.where(valid_s, vy_s, 0)**2)
    im = axes[1,1].imshow(np.where(valid_s, speed_s, np.nan), cmap='magma', vmin=0, vmax=5000)
    axes[1,1].set_title('Speed (m/yr)'); plt.colorbar(im, ax=axes[1,1])
    
    if valid_px > 0:
        speed = np.sqrt(vx[valid]**2 + vy[valid]**2)
        print(f"\nSpeed stats:")
        for p in [50, 75, 90, 95, 99]:
            print(f"  P{p}: {np.percentile(speed, p):.0f} m/yr")
    
    if 'ex' in data:
        err_s = data['ex'][::step, ::step]
        valid_err_s = ~np.isnan(err_s) & (err_s > 0) & valid_s
        im = axes[1,2].imshow(np.where(valid_err_s, err_s, np.nan), cmap='YlOrRd', vmin=0, vmax=500)
        axes[1,2].set_title('ex (error x)'); plt.colorbar(im, ax=axes[1,2])
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'spatial_coverage.png'), dpi=150, bbox_inches='tight')
    plt.show()
    return valid

if sample_data:
    valid_mask = analyze_spatial_coverage(sample_data, title=f"Cycle: {sample_key}")


## 4) Temporal Completeness Analysis


In [ ]:
def analyze_temporal_coverage(catalog):
    dates_start = sorted([v['date_start'] for v in catalog.values()])
    print(f"Total cycles: {len(catalog)}")
    print(f"Date range: {dates_start[0].strftime('%Y-%m-%d')} → {dates_start[-1].strftime('%Y-%m-%d')}")
    
    if len(dates_start) > 1:
        gaps = [(dates_start[i+1] - dates_start[i]).days for i in range(len(dates_start)-1)]
        print(f"Gap stats: min={min(gaps)}d, max={max(gaps)}d, median={np.median(gaps):.0f}d")
    
    by_year = defaultdict(int)
    for d in dates_start: by_year[d.year] += 1
    print(f"\nCycles per year:")
    for yr in sorted(by_year.keys()): print(f"  {yr}: {by_year[yr]}")
    
    by_month = defaultdict(int)
    for d in dates_start: by_month[d.strftime('%Y-%m')] += 1
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    axes[0].scatter(dates_start, [1]*len(dates_start), marker='|', s=200, color='steelblue')
    axes[0].set_title('Velocity Cycle Timeline')
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    axes[0].set_yticks([])
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)
    
    months = sorted(by_month.keys())
    counts = [by_month[m] for m in months]
    axes[1].bar(range(len(months)), counts, color='steelblue')
    step = max(1, len(months)//20)
    axes[1].set_xticks(range(0, len(months), step))
    axes[1].set_xticklabels([months[i] for i in range(0, len(months), step)], rotation=45)
    axes[1].set_title('Cycles per Month'); axes[1].set_ylabel('Count')
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'temporal_coverage.png'), dpi=150, bbox_inches='tight')
    plt.show()

analyze_temporal_coverage(cycle_catalog)


## 5) Data Quality Checks


In [ ]:
def quality_checks(data):
    if data is None or 'vx' not in data or 'vy' not in data:
        print("Need vx, vy"); return
    
    vx, vy = data['vx'], data['vy']
    valid = ~np.isnan(vx) & ~np.isnan(vy) & (np.abs(vx) < 50000) & (np.abs(vy) < 50000)
    speed = np.sqrt(np.where(valid, vx, 0)**2 + np.where(valid, vy, 0)**2)
    
    print("=== Quality Report ===")
    print(f"Total pixels: {vx.size:,}")
    print(f"Valid pixels: {valid.sum():,} ({valid.sum()/vx.size*100:.1f}%)")
    print(f"NaN pixels:   {np.isnan(vx).sum():,}")
    print(f"Zero pixels:  {((vx[valid] == 0) & (vy[valid] == 0)).sum():,}")
    
    for thresh in [1000, 5000, 10000, 20000]:
        n = (speed[valid] > thresh).sum()
        print(f"Speed > {thresh}: {n:,} ({n/max(1,valid.sum())*100:.2f}%)")
    
    if 'ex' in data:
        err = data['ex']
        valid_err = valid & ~np.isnan(err) & (err > 0) & (err < 50000)
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].hist(err[valid_err].ravel()[::max(1, valid_err.sum()//50000)],
                     bins=200, log=True, color='coral', edgecolor='none')
        axes[0].set_title('Error Distribution'); axes[0].set_xlabel('m/yr')
        if valid_err.any():
            axes[0].axvline(np.median(err[valid_err]), color='k', ls='--',
                           label=f'median={np.median(err[valid_err]):.0f}')
            axes[0].legend()
        
        n_sub = min(50000, valid_err.sum())
        if n_sub > 0:
            idx = np.random.choice(np.where(valid_err.ravel())[0], n_sub, replace=False)
            axes[1].scatter(speed.ravel()[idx], err.ravel()[idx], s=1, alpha=0.1, color='coral')
            axes[1].set_xlabel('Speed (m/yr)'); axes[1].set_ylabel('Error (m/yr)')
            axes[1].set_title('Error vs Speed')
            axes[1].set_xlim(0, 10000); axes[1].set_ylim(0, 1000)
        
        plt.tight_layout()
        plt.savefig(os.path.join(FIGURES_DIR, 'quality_checks.png'), dpi=150, bbox_inches='tight')
        plt.show()

if sample_data:
    quality_checks(sample_data)


## 6) Convert All Cycles to `.npz`


In [ ]:
def convert_cycle_to_npz(cycle_key, cycle_info, output_dir=OUTPUT_DIR):
    """
    Load one cycle (full raster), check for zeros, save as .npz.
    Returns (output_path, status).
    """
    try:
        data = load_cycle_full(cycle_info, verbose=False)
    except Exception as e:
        return None, f'error: {e}'
    
    if data is None:
        return None, 'skipped_empty'
    
    vx, vy = data['vx'], data['vy']
    
    valid = ~np.isnan(vx) & ~np.isnan(vy)
    valid &= ~np.isinf(vx) & ~np.isinf(vy)
    valid &= (np.abs(vx) < 50000) & (np.abs(vy) < 50000)
    
    vx_clean = np.where(valid, vx, 0.0).astype(np.float32)
    vy_clean = np.where(valid, vy, 0.0).astype(np.float32)
    
    ex = data.get('ex')
    ey = data.get('ey')
    if ex is None: ex = np.zeros_like(vx)
    if ey is None: ey = np.zeros_like(vy)
    ex_clean = np.where(valid & ~np.isnan(ex) & (ex > 0), ex, 0.0).astype(np.float32)
    ey_clean = np.where(valid & ~np.isnan(ey) & (ey > 0), ey, 0.0).astype(np.float32)
    
    t = data['transform']
    meta = {
        'source': 'NSIDC-0766',
        'crs': data['crs'],
        'resolution_m': TARGET_RES,
        'units': 'm/yr',
        'shape': list(vx.shape),
        'valid_fraction': float(valid.sum() / valid.size),
        'date_start': data['date_start'].strftime('%Y-%m-%d'),
        'date_end': data['date_end'].strftime('%Y-%m-%d'),
        'transform': [t.a, t.b, t.c, t.d, t.e, t.f],
        'bounds': list(data['bounds']),
        'roi_lonlat': ROI_BOUNDS,
    }
    
    d1 = data['date_start'].strftime('%Y%m%d')
    d2 = data['date_end'].strftime('%Y%m%d')
    out_path = os.path.join(output_dir, f"velocity_{d1}_{d2}.npz")
    
    np.savez_compressed(out_path,
                        vx=vx_clean, vy=vy_clean,
                        ex=ex_clean, ey=ey_clean,
                        valid_mask=valid,
                        metadata=json.dumps(meta))
    
    return out_path, 'saved'


def convert_all_cycles(catalog, output_dir=OUTPUT_DIR):
    print(f"Converting {len(catalog)} cycles to .npz...")
    saved, skipped, errs = [], [], []
    
    for ck, ci in tqdm(catalog.items(), desc="Converting"):
        path, status = convert_cycle_to_npz(ck, ci, output_dir)
        if status == 'saved': saved.append(path)
        elif status == 'skipped_empty': skipped.append(ck)
        else: errs.append((ck, status))
        gc.collect()
    
    print(f"\n{'='*60}")
    print(f"Saved:   {len(saved)}")
    print(f"Skipped: {len(skipped)} (all-zero / no data)")
    print(f"Errors:  {len(errs)}")
    
    if skipped:
        print(f"\nSkipped cycles:")
        for k in skipped: print(f"  {k}")
    if errs:
        print(f"\nErrors:")
        for k, e in errs: print(f"  {k}: {e}")
    if saved:
        total_mb = sum(os.path.getsize(p) for p in saved) / 1e6
        print(f"\nTotal size: {total_mb:.1f} MB ({os.path.getsize(saved[0])/1e6:.1f} MB each)")
    
    return saved, skipped, errs

saved_paths, skipped_empty, errors = convert_all_cycles(cycle_catalog)


In [ ]:
def verify_npz(npz_path):
    data = np.load(npz_path, allow_pickle=True)
    print(f"\nVerifying: {Path(npz_path).name}")
    for key in data.files:
        arr = data[key]
        if key == 'metadata':
            meta = json.loads(str(arr))
            print(f"  metadata: {json.dumps(meta, indent=2)}")
        else:
            print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}, "
                  f"range=[{arr.min():.1f}, {arr.max():.1f}]")
    data.close()

npz_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "velocity_*.npz")))
print(f"Total .npz files: {len(npz_files)}")
if npz_files:
    verify_npz(npz_files[0])
    if len(npz_files) > 1:
        verify_npz(npz_files[-1])


## Summary
